In [ ]:
storage_account = "stretailbidevgk01"
container = "retail-data"

storage_key = dbutils.secrets.get(
    scope="kv-retail-bi-dev",
    key="adls-storage-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

print("ADLS authentication configured successfully")

## **Orderlines**

In [ ]:

orderlines_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "bronze/wwi/orderlines/"
)

df_orderlines_raw = (
    spark.read
    .format("delta")
    .load(orderlines_path)
)

df_orderlines_raw.printSchema()
display(df_orderlines_raw.limit(5))

In [ ]:
""" old code
orderlines_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "landing/wwi/orderlines/"
)


df_orderlines = spark.read.option("header", "true").csv(orderlines_path)

display(df_orderlines)
df_orderlines_raw = spark.read \
    .option("header", "false") \
    .csv(orderlines_path)
"""

orderline_columns = [
    "OrderLineID",
    "OrderID",
    "StockItemID",
    "Description",
    "PackageTypeID",
    "Quantity",
    "UnitPrice",
    "TaxRate",
    "PickedQuantity",
    "PickingCompletedWhen",
    "LastEditedBy",
    "LastEditedWhen",
    "_ingested_at",
    "_source_file",
]




df_orderlines_raw = (
    spark.read
    .format("delta")
    .load(orderlines_path)
)


df_orderlines = df_orderlines_raw.toDF(*orderline_columns)

display(df_orderlines)


In [ ]:

"""orderlines_raw_check = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "false")
    .csv(orderlines_path)
)

print("Column count:", len(orderlines_raw_check.columns))
print("Columns:", orderlines_raw_check.columns)

display(orderlines_raw_check.limit(5))"""

In [ ]:
"""first_row = orderlines_raw_check.first()

print("Number of values in first row:", len(first_row))
print("First row:")
print(first_row)"""

In [ ]:
print("Raw columns:", len(df_orderlines_raw.columns))
print("Renamed columns:", len(df_orderlines.columns))
print(df_orderlines.columns)

In [ ]:
df_orderlines.printSchema()

In [ ]:
"""Cast the columns - from raw/bronze → clean Silver by applying the correct data types"""
from pyspark.sql.functions import col, to_timestamp

df_orderlines_silver = (
    df_orderlines
    .select(
        col("OrderLineID").cast("int").alias("OrderLineID"),
        col("OrderID").cast("int").alias("OrderID"),
        col("StockItemID").cast("int").alias("StockItemID"),
        col("Description").alias("Description"),
        col("PackageTypeID").cast("int").alias("PackageTypeID"),
        col("Quantity").cast("int").alias("Quantity"),
        col("UnitPrice").cast("decimal(18,2)").alias("UnitPrice"),
        col("TaxRate").cast("decimal(5,2)").alias("TaxRate"),
        col("PickedQuantity").cast("int").alias("PickedQuantity"),
        to_timestamp(
            col("PickingCompletedWhen"),
            "yyyy-MM-dd HH:mm:ss.SSSSSSS"
        ).alias("PickingCompletedWhen"),
        col("LastEditedBy").cast("int").alias("LastEditedBy"),
        to_timestamp(
            col("LastEditedWhen"),
            "yyyy-MM-dd HH:mm:ss.SSSSSSS"
        ).alias("LastEditedWhen"),
             to_timestamp(
            col("_ingested_at"),
            "yyyy-MM-dd HH:mm:ss.SSSSSSS"
        ).alias("_ingested_at"),
              col("_source_file").alias("_source_file")
    )
)

display(df_orderlines_silver)

In [ ]:
df_orderlines_silver.printSchema()

In [ ]:
"""data-quality validation"""
from pyspark.sql.functions import col, count, when

print("Total rows:", df_orderlines_silver.count())

print("\nNull counts:")
display(
    df_orderlines_silver.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df_orderlines_silver.columns
    ])
)
print(
    "Duplicate OrderLineIDs:",
    df_orderlines_silver.groupBy("OrderLineID")
    .count()
    .filter(col("count") > 1)
    .count()
)
print("Invalid Quantity:", 
      df_orderlines_silver.filter(col("Quantity") <= 0).count())

print("Invalid UnitPrice:", 
      df_orderlines_silver.filter(col("UnitPrice") < 0).count())

print("Invalid TaxRate:", 
      df_orderlines_silver.filter(col("TaxRate") < 0).count())

print("PickedQuantity > Quantity:",
      df_orderlines_silver.filter(
          col("PickedQuantity") > col("Quantity")
      ).count())

In [ ]:
"""null-count output"""
from pyspark.sql.functions import col, count, when

display(
    df_orderlines_silver.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df_orderlines_silver.columns
    ])
)

In [ ]:
"""write the Silver layer to ADLS"""
silver_orderlines_path = "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/silver/wwi/orderlines/"
print(dbutils.fs.ls("abfss://retail-data@stretailbidevgk01.dfs.core.windows.net"))

In [ ]:
configs = [
    k for k in spark.conf.getAll
    if "azure.account" in k.lower()
    or "oauth" in k.lower()
    or "dfs" in k.lower()
]

for k in configs:
    print(k)

In [ ]:
print([
    k for k in spark.conf.getAll
    if "fs.azure.account.key" in k.lower()
])

In [ ]:
display(
    dbutils.fs.ls(
        "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    )
)

In [ ]:
silver_orderlines_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "silver/wwi/orderlines/"
)

In [ ]:
"""df_orderlines_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_orderlines_path)"""

df_orderlines_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_orderlines_path)

## **Orders**

In [ ]:

orders_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "bronze/wwi/orders/"
)

df_orders_raw = (
    spark.read
    .format("delta")
    .load(orders_path)
)

df_orders_raw.printSchema()
display(df_orders_raw.limit(5))


In [ ]:
"""**** Orders ******"""

print("Column count:", len(df_orders_raw.columns))
print("Columns:", df_orders_raw.columns)

first_row = df_orders_raw.first()
print("Number of values in first row:", len(first_row))
print("First row:")
print(first_row)


In [ ]:
order_columns = [
    'OrderID',
    'CustomerID',
    'SalespersonPersonID',
    'PickedByPersonID',
    'ContactPersonID',
    'OrderDate',
    'ExpectedDeliveryDate',
    'CustomerPurchaseOrderNumber',
    'IsUndersupplyBackordered',
    'PickingCompletedWhen',
    'Comments',
    'DeliveryInstructions',
    'InternalComments',
    'TotalDryItems',
    'TotalChillerItems',
    'LastEditedWhen',
    '_ingested_at',
    '_source_file'
]

df_orders = df_orders_raw.toDF(*order_columns)

print("Column count:", len(df_orders.columns))
print("Columns:", df_orders.columns)

display(df_orders.limit(5))

In [ ]:
from pyspark.sql.functions import col

df_orders_silver = (
    df_orders_raw
    .select(
        col("OrderID").cast("int").alias("OrderID"),
        col("CustomerID").cast("int").alias("CustomerID"),
        col("SalespersonPersonID").cast("int").alias("SalespersonPersonID"),
        col("PickedByPersonID").cast("int").alias("PickedByPersonID"),
        col("ContactPersonID").cast("int").alias("ContactPersonID"),
        col("OrderDate").cast("timestamp"),
        col("ExpectedDeliveryDate").cast("timestamp"),
        col("CustomerPurchaseOrderNumber").cast("string").alias("CustomerPurchaseOrderNumber"),
        col("IsUndersupplyBackordered").cast("boolean").alias("IsUndersupplyBackordered"),
        col("PickingCompletedWhen").cast("timestamp"), 
        col("Comments").cast("string").alias("Comments"),
        col("DeliveryInstructions").cast("string").alias("DeliveryInstructions"),
        col("InternalComments").cast("string").alias("InternalComments"),
        col("TotalDryItems").cast("int").alias("TotalDryItems"),
        col("TotalChillerItems").cast("int").alias("TotalChillerItems"),
        col("LastEditedWhen").cast("timestamp"),
        col("_ingested_at").cast("timestamp"),           
        col("_source_file").alias("_source_file")
    )
)

df_orders_silver.printSchema()

In [ ]:
""" Row count + nulls """

from pyspark.sql.functions import col, count, when, lit, sum as spark_sum

print("Total Orders:", df_orders_silver.count())

null_counts = df_orders_silver.select([
    spark_sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df_orders_silver.columns
])

display(null_counts)

In [ ]:
"""  Duplicate OrderIDs """

print(
    "Duplicate OrderIDs:",
    df_orders_silver
        .groupBy("OrderID")
        .count()
        .filter(col("count") > 1)
        .count()
)
"""  Business-rule checks """
print(
    "Invalid OrderID:",
    df_orders_silver.filter(col("OrderID") <= 0).count()
)

print(
    "Invalid CustomerID:",
    df_orders_silver.filter(col("CustomerID") <= 0).count()
)

print(
    "ExpectedDeliveryDate before OrderDate:",
    df_orders_silver.filter(
        col("ExpectedDeliveryDate") < col("OrderDate")
    ).count()
)

print(
    "PickingCompletedWhen before OrderDate:",
    df_orders_silver.filter(
        col("PickingCompletedWhen") < col("OrderDate")
    ).count()
)

"""   """

In [ ]:
"""  ****Orders to Silver Delta***"""
silver_orders_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "silver/wwi/orders/"
)
df_orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_orders_path)
    
df_orders_check = (
    spark.read
    .format("delta")
    .load(silver_orders_path)
)

print("Silver Orders row count:", df_orders_check.count())
print("Silver Orders column count:", len(df_orders_check.columns))

display(df_orders_check.limit(10))

## Customer

In [ ]:


customers_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "bronze/wwi/customers/"
)

df_customers_raw = (
    spark.read
    .format("delta")
    .load(customers_path)
)

df_customers_raw.printSchema()
display(df_customers_raw.limit(5))

In [ ]:
"""****Customers data****"""
"""customers_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "landing/wwi/customers/"
)

df_customers_raw = (
    spark.read
    .option("header", "false")
    .csv(customers_path)
)"""

print("Column count:", len(df_customers_raw.columns))
print("Columns:", df_customers_raw.columns)

first_row = df_customers_raw.first()

print("Number of values in first row:", len(first_row))
print("First row:")
print(first_row)


In [ ]:
display(df_customers_raw.limit(5))

In [ ]:

""" Rename customer columns """
customer_columns = [
    "CustomerID",
    "CustomerName",
    "BillToCustomerID",
    "CustomerCategoryID",
    "BuyingGroupID",
    "PrimaryContactPersonID",
    "AlternateContactPersonID",
    "DeliveryMethodID",
    "DeliveryCityID",
    "PostalCityID",
    "CreditLimit",
    "AccountOpenedDate",
    "StandardDiscountPercentage",
    "IsStatementSent",
    "IsOnCreditHold",
    "PaymentDays",
    "PhoneNumber",
    "FaxNumber",
    "DeliveryRun",
    "RunPosition",
    "WebsiteURL",
    "DeliveryAddressLine1",
    "DeliveryAddressLine2",
    "DeliveryPostalCode",
    "DeliveryLocation",
    "PostalAddressLine1",
    "PostalAddressLine2",
    "PostalPostalCode",
    "LastEditedBy",
    "ValidFrom",
    "ValidTo",
    "_ingested_at",
    "_source_file"
]

df_customers = df_customers_raw.toDF(*customer_columns)

print("Column count:", len(df_customers.columns))
print("Columns:", df_customers.columns)

display(df_customers.limit(5))

In [ ]:
"""  Cast Customer columns """
from pyspark.sql.functions import col

df_customers_silver = (
    df_customers
    .select(
        col("CustomerID").cast("int"),
        col("CustomerName").cast("string"),
        col("BillToCustomerID").cast("int"),
        col("CustomerCategoryID").cast("int"),
        col("BuyingGroupID").cast("int"),
        col("PrimaryContactPersonID").cast("int"),
        col("AlternateContactPersonID").cast("int"),
        col("DeliveryMethodID").cast("int"),
        col("DeliveryCityID").cast("int"),
        col("PostalCityID").cast("int"),
        col("CreditLimit").cast("decimal(18,2)"),
        col("AccountOpenedDate").cast("timestamp"),
        col("StandardDiscountPercentage").cast("decimal(18,3)"),
        col("IsStatementSent").cast("boolean"),
        col("IsOnCreditHold").cast("boolean"),
        col("PaymentDays").cast("int"),
        col("PhoneNumber").cast("string"),
        col("FaxNumber").cast("string"),
        col("DeliveryRun").cast("string"),
        col("RunPosition").cast("int"),
        col("WebsiteURL").cast("string"),
        col("DeliveryAddressLine1").cast("string"),
        col("DeliveryAddressLine2").cast("string"),
        col("DeliveryPostalCode").cast("string"),
        col("DeliveryLocation").cast("string"),
        col("PostalAddressLine1").cast("string"),
        col("PostalAddressLine2").cast("string"),
        col("PostalPostalCode").cast("string"),
        col("LastEditedBy").cast("int"),
        col("ValidFrom").cast("timestamp"),
        col("ValidTo").cast("timestamp"),
        col("_ingested_at").cast("timestamp"),
        col("_source_file").cast("string")
    )
)

print("Column count:", len(df_customers_silver.columns))
print("Row count:", df_customers_silver.count())

df_customers_silver.printSchema()
display(df_customers_silver.limit(5))

In [ ]:
""" Customer data-quality checks """
from pyspark.sql.functions import count

# 1. Duplicate CustomerID
duplicate_customers = (
    df_customers_silver
    .groupBy("CustomerID")
    .count()
    .filter(col("count") > 1)
    .count()
)

# 2. Invalid CustomerID
invalid_customer_id = (
    df_customers_silver
    .filter(col("CustomerID").isNull() | (col("CustomerID") <= 0))
    .count()
)

# 3. Invalid PaymentDays
invalid_payment_days = (
    df_customers_silver
    .filter(col("PaymentDays").isNull() | (col("PaymentDays") < 0))
    .count()
)

# 4. Invalid discount percentage
invalid_discount = (
    df_customers_silver
    .filter(
        col("StandardDiscountPercentage").isNull()
        | (col("StandardDiscountPercentage") < 0)
        | (col("StandardDiscountPercentage") > 100)
    )
    .count()
)

# 5. Invalid ValidFrom / ValidTo
invalid_valid_dates = (
    df_customers_silver
    .filter(col("ValidTo") < col("ValidFrom"))
    .count()
)

print("Duplicate CustomerIDs:", duplicate_customers)
print("Invalid CustomerIDs:", invalid_customer_id)
print("Invalid PaymentDays:", invalid_payment_days)
print("Invalid Discount %:", invalid_discount)
print("Invalid ValidFrom/ValidTo:", invalid_valid_dates)

In [ ]:
"""  Cast Customer columns """ 
from pyspark.sql.functions import col 
df_customers_silver = (df_customers.select(col("CustomerID").cast("int"),
                                           col("CustomerName").cast("string"),
                                           col("BillToCustomerID").cast("int"),
                                           col("CustomerCategoryID").cast("int"),
                                           col("BuyingGroupID").cast("int"),
                                           col("PrimaryContactPersonID").cast("int"),
                                           col("AlternateContactPersonID").cast("int"),
                                           col("DeliveryMethodID").cast("int"),
                                           col("DeliveryCityID").cast("int"),
                                           col("PostalCityID").cast("int"),
                                           col("CreditLimit").cast("decimal(18,2)"),
                                           col("AccountOpenedDate").cast("timestamp"),
                                           col("StandardDiscountPercentage").cast("decimal(18,3)"),
                                           col("IsStatementSent").cast("boolean"),
                                           col("IsOnCreditHold").cast("boolean"),
                                           col("PaymentDays").cast("int"),
                                           col("PhoneNumber").cast("string"),
                                           col("FaxNumber").cast("string"),
                                           col("DeliveryRun").cast("string"),
                                           col("RunPosition").cast("int"),
                                           col("WebsiteURL").cast("string"),
                                           col("DeliveryAddressLine1").cast("string"),
                                           col("DeliveryAddressLine2").cast("string"),
                                           col("DeliveryPostalCode").cast("string"),
                                           col("DeliveryLocation").cast("string"),
                                           col("PostalAddressLine1").cast("string"),
                                            col("PostalAddressLine2").cast("string"),
                                            col("PostalPostalCode").cast("string"),
                                            col("LastEditedBy").cast("int"),
                                            col("ValidFrom").cast("timestamp"),
                                            col("ValidTo").cast("timestamp"),
                                            col("_ingested_at").cast("timestamp"),
                                            col("_source_file").cast("string")
                                             )
                        )
print("Column count:", len(df_customers_silver.columns)) 
print("Row count:", df_customers_silver.count()) 
df_customers_silver.printSchema() 
display(df_customers_silver.limit(5))

Clean Customer string fields

In [ ]:
from pyspark.sql.functions import trim, upper, when, lit

df_customers_silver = (
    df_customers_silver

    .withColumn("CustomerName", trim(col("CustomerName")))
    .withColumn("PhoneNumber", trim(col("PhoneNumber")))
    .withColumn("FaxNumber", trim(col("FaxNumber")))
    .withColumn("WebsiteURL", trim(col("WebsiteURL")))
    .withColumn("DeliveryAddressLine1", trim(col("DeliveryAddressLine1")))
    .withColumn("DeliveryAddressLine2", trim(col("DeliveryAddressLine2")))
    .withColumn("DeliveryPostalCode", trim(col("DeliveryPostalCode")))
    .withColumn("PostalAddressLine1", trim(col("PostalAddressLine1")))
    .withColumn("PostalAddressLine2", trim(col("PostalAddressLine2")))
    .withColumn("PostalPostalCode", trim(col("PostalPostalCode")))
)

Remove duplicate customers

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

customer_window = (
    Window
    .partitionBy("CustomerID")
    .orderBy(col("ValidFrom").desc_nulls_last())
)

df_customers_silver = (
    df_customers_silver
    .withColumn("_rn", row_number().over(customer_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

print("Row count after deduplication:", df_customers_silver.count())

Add Silver processing timestamp

In [ ]:
from pyspark.sql.functions import current_timestamp

df_customers_silver = (
    df_customers_silver
    .withColumn("silver_processed_timestamp", current_timestamp())
)

Customer DQ check

In [ ]:
print("=== CUSTOMER SILVER DQ CHECK ===")

print(
    "Null CustomerID:",
    df_customers_silver.filter(col("CustomerID").isNull()).count()
)

print(
    "Duplicate CustomerID:",
    df_customers_silver
        .groupBy("CustomerID")
        .count()
        .filter(col("count") > 1)
        .count()
)

print(
    "Null CustomerName:",
    df_customers_silver.filter(col("CustomerName").isNull()).count()
)

print(
    "Negative CreditLimit:",
    df_customers_silver
        .filter(col("CreditLimit") < 0)
        .count()
)

print(
    "Invalid Discount:",
    df_customers_silver
        .filter(
            (col("StandardDiscountPercentage") < 0) |
            (col("StandardDiscountPercentage") > 100)
        )
        .count()
)

print(
    "Invalid Account Dates:",
    df_customers_silver
        .filter(
            col("ValidTo").isNotNull() &
            (col("ValidFrom") > col("ValidTo"))
        )
        .count()
)

schema/count validation

In [ ]:
print("=== FINAL CUSTOMER SILVER ===")

print("Column count:", len(df_customers_silver.columns))
print("Row count:", df_customers_silver.count())

df_customers_silver.printSchema()

display(
    df_customers_silver
    .orderBy("CustomerID")
    .limit(10)
)

Customer Silver

In [ ]:
silver_customers_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "silver/wwi/customers"
)

(
    df_customers_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_customers_path)
)

print("Customer Silver written successfully.")

Read back and verify

In [ ]:
df_customers_silver_check = (
    spark.read
    .format("delta")
    .load(silver_customers_path)
)

print("Silver Customer row count:", df_customers_silver_check.count())
print("Silver Customer column count:", len(df_customers_silver_check.columns))

display(df_customers_silver_check.limit(5))